## LASSO ON CLASSIFICATION

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from scipy.sparse import issparse

Fetching OpenML dataset 1104...
   Preprocessing (Log2 + Scaling)...
   Training set: (57, 7129)

--- 🔵 1. Baseline Random Forest ---
Baseline Accuracy: 0.9333 (All 7129 genes)

--- 🔴 2. LASSO Selection (The Oracle) ---
LASSO selected 19 features.
Reduction: 99.7%

--- 🟡 3. Training RF on 19 LASSO genes ---
LASSO RF Accuracy: 0.9333

--- 📊 Final Scoreboard ---
Baseline: 0.9333
LASSO:    0.9333
✅ LASSO maintained/improved accuracy (Standard behavior).


In [ ]:
DATASET_ID = 1104 
C_PARAM = 0.4      # Controls LASSO strictness (Lower = Fewer features)

print(f"Fetching OpenML dataset {DATASET_ID}...")
dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
X_raw = dataset.data
y_raw = dataset.target

if issparse(X_raw):
    X_dense = pd.DataFrame(X_raw.toarray())
else:
    X_dense = pd.DataFrame(X_raw).select_dtypes(include=[np.number])

In [ ]:
print("   Preprocessing (Log2 + Scaling)...")
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_dense)
X_log = np.log2(X_imputed - np.min(X_imputed) + 1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

le = LabelEncoder()
y_numeric = le.fit_transform(y_raw)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_numeric, test_size=0.2, random_state=42, stratify=y_numeric
)

print(f"   Training set: {X_train.shape}")

rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)

acc_base = accuracy_score(y_test, rf_base.predict(X_test))
print(f"Baseline Accuracy: {acc_base:.4f} (All {X_train.shape[1]} genes)")


In [ ]:
# We use Logistic Regression with L1 penalty to select features
lasso = LogisticRegression(
    penalty='l1', 
    C=0.4, 
    solver='liblinear', 
    random_state=42, 
    max_iter=5000
)

lasso.fit(X_train, y_train)

In [ ]:
# Select features where coef != 0
model = SelectFromModel(lasso, prefit=True)
X_train_lasso = model.transform(X_train)
X_test_lasso = model.transform(X_test)
n_features = X_train_lasso.shape[1]

print(f"LASSO selected {n_features} features.")
print(f"Reduction: {100 - (n_features/X_train.shape[1]*100):.1f}%")

In [ ]:
rf_lasso = RandomForestClassifier(n_estimators=100, random_state=42)
rf_lasso.fit(X_train_lasso, y_train)

acc_lasso = accuracy_score(y_test, rf_lasso.predict(X_test_lasso))

print(f"LASSO RF Accuracy: {acc_lasso:.4f}")
print(f"Baseline: {acc_base:.4f}")
print(f"LASSO:    {acc_lasso:.4f}")

## BOTH CSV AND OPENML CODE

In [ ]:
# setting up variables and parameters for the pipeline
SOURCE_TYPE = "CSV"        
FILE_PATH = "datasets/df_destx.csv"
SEP = ","

DATASET_ID = 1104            # Used only if SOURCE_TYPE = OPENML
C_PARAM = 0.2
TEST_SIZE = 0.2
RANDOM_STATE = 42


Starting LASSO Pipeline [CSV]...
Raw CSV shape: (167, 726)
Final Feature Shape: (167, 717)
Training shape: (133, 717)
Baseline Accuracy: 0.9706 (All 717 features)
LASSO selected 23 features.
Reduction: 96.8%
LASSO RF Accuracy: 1.0000
Baseline: 0.9706
LASSO:    1.0000


In [ ]:
print(f"Starting LASSO Pipeline [{SOURCE_TYPE}]...")

if SOURCE_TYPE == "OPENML":
    dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser="auto")
    X_raw = dataset.data
    y_raw = dataset.target

    if issparse(X_raw):
        X_df = pd.DataFrame(X_raw.toarray())
    else:
        X_df = pd.DataFrame(X_raw)

    y_series = pd.Series(y_raw)

elif SOURCE_TYPE == "CSV":
    df = pd.read_csv(FILE_PATH, sep=SEP)
    print(f"Raw CSV shape: {df.shape}")

    # Detect label column automatically
    label_col = None
    for col in ["type", "class", "label", "phenotype", "target", "group"]:
        if col in df.columns:
            label_col = col
            break

    if label_col is None:
        raise ValueError("No label column found in CSV.")

    y_series = df[label_col]
    X_df = df.drop(columns=[label_col])

In [ ]:

# Drop non-numeric safely
X_numeric = X_df.apply(pd.to_numeric, errors="coerce")
X_numeric = X_numeric.select_dtypes(include=[np.number]).dropna(axis=1, how="all")

# Transpose if genes are rows
if X_numeric.shape[0] > X_numeric.shape[1]:
    X_numeric = X_numeric.T
    print("Transposed to Samples x Features")

In [ ]:
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X_numeric)

# Log transform only if data is positive
if np.min(X_imputed) >= 0:
    X_log = np.log2(X_imputed + 1)
else:
    X_log = X_imputed

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y_series.astype(str))



X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

In [ ]:
print(f"Training shape: {X_train.shape}")


rf_base = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_base.fit(X_train, y_train)

acc_base = accuracy_score(y_test, rf_base.predict(X_test))
print(f"Baseline Accuracy: {acc_base:.4f} (All {X_train.shape[1]} features)")

In [ ]:
lasso = LogisticRegression(
    penalty="l1",
    C=C_PARAM,
    solver="liblinear",
    random_state=RANDOM_STATE,
    max_iter=5000
)

lasso.fit(X_train, y_train)

selector = SelectFromModel(lasso, prefit=True)
X_train_lasso = selector.transform(X_train)
X_test_lasso = selector.transform(X_test)

In [ ]:

n_features = X_train_lasso.shape[1]

print(f"LASSO selected {n_features} features.")
print(f"Reduction: {100 - (n_features / X_train.shape[1] * 100):.1f}%")
rf_lasso = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_lasso.fit(X_train_lasso, y_train)

acc_lasso = accuracy_score(y_test, rf_lasso.predict(X_test_lasso))

print(f"LASSO RF Accuracy: {acc_lasso:.4f}")
print(f"Baseline: {acc_base:.4f}")
print(f"LASSO:    {acc_lasso:.4f}")